# Week 34, Day 5: Refactoring step by step

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w034d5.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
def f(d, b):
    if d > 0:
        if b > 0:
            x = d * 20 * b
            if x > 500:
                x = 500
            return x
    return 0

DAILY_FEE = 20
FEE_CAP = 500

def late_fee(days_late, books):
    if days_late <= 0 or books <= 0:
        return 0
    return min(days_late * DAILY_FEE * books, FEE_CAP)

cases = [(3, 2), (0, 5), (40, 1), (2, -1)]
print([f(d, b) for d, b in cases])
print([late_fee(d, b) for d, b in cases])

In [ ]:
def shipping_cost(grams, country, express=False):
    if country == "UK":
        cost = 300 + 100 * ((grams - 1) // 1000)
    else:
        cost = 1500
    if express:
        cost = cost * 2
    if grams > 20000:
        cost = None
    return cost

# Characterisation: try ordinary cases, boundaries and odd inputs, and write down what happens.
for args in [(500, "UK"), (1000, "UK"), (1001, "UK"), (500, "FR"), (500, "UK", True),
             (20000, "UK"), (20001, "UK"), (500, "uk"), (25000, "FR", True)]:
    print(args, "->", shipping_cost(*args))

In [ ]:
def discount_nested(customer, total):
    if customer is not None:
        if total > 0:
            if customer["member"]:
                if total >= 5000:
                    return total // 10
                else:
                    return total // 20
            else:
                return 0
        else:
            return 0
    else:
        return 0

def discount(customer, total):
    if customer is None or total <= 0:
        return 0
    if not customer["member"]:
        return 0
    if total >= 5000:
        return total // 10
    return total // 20

cases = [(None, 100), ({"member": True}, 6000), ({"member": True}, 1000), ({"member": False}, 9000), ({"member": True}, 0)]
print([discount_nested(c, t) for c, t in cases] == [discount(c, t) for c, t in cases], [discount(c, t) for c, t in cases])

> **Key idea.** Refactoring (Martin Fowler) changes code's structure without changing its observable behaviour, to make it easier to understand and change; keep it separate from adding features (Kent Beck's two hats). Before refactoring untested code, write characterisation tests (Michael Feathers) that record what it does now, oddities included. Use named small steps: rename, constants for magic numbers, extract variable or function, inline, guard clauses, parameter objects, tables or polymorphism for conditionals, split loop. Test after every step, commit when green, undo when red. Refactor to prepare for a change, while reading, and on the third repetition.

### Exercise: Pinning down legacy behaviour

`legacy.py` holds `shipping_cost`, exactly as in the lesson. Before anyone refactors it, write pytest **characterisation tests** that record what it does, including its odd behaviour. Your tests must pass on the real code and catch all five broken versions, each of which changes behaviour in a way a careless refactoring might.

In [ ]:
# Run this cell first: it creates the files used below.
with open('legacy.py', 'w', encoding='utf-8') as f:
    f.write('def shipping_cost(grams, country, express=False):\n    if country == "UK":\n        cost = 300 + 100 * ((grams - 1) // 1000)\n    else:\n        cost = 1500\n    if express:\n        cost = cost * 2\n    if grams > 20000:\n        cost = None\n    return cost\n')

In [ ]:
from legacy import shipping_cost

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"legacy.py\").read()\nBROKEN = {\n    \"first band boundary\": GOOD.replace(\"((grams - 1) // 1000)\", \"(grams // 1000)\"),\n    \"express triples\": GOOD.replace(\"cost = cost * 2\", \"cost = cost * 3\"),\n    \"case-insensitive country\": GOOD.replace('if country == \"UK\":', 'if country.upper() == \"UK\":'),\n    \"heavy limit moved\": GOOD.replace(\"if grams > 20000:\", \"if grams >= 20000:\"),\n    \"limit checked first\": GOOD.replace(\"    if express:\\n        cost = cost * 2\\n    if grams > 20000:\\n        cost = None\\n\", \"    if grams > 20000:\\n        cost = None\\n    if express:\\n        cost = cost * 2\\n\"),\n}\ndef test_catches():\n    \"your tests pass on the real code and catch all five broken versions\"\n    results = against(source, \"legacy\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"legacy.py\", GOOD)\n    assert results[\"correct\"] == [], f\"these fail on the real code: {results['correct']}\"\n    missed = [k for k in BROKEN if not results[k]]\n    assert not missed, f\"not caught: {missed}\"")

### Exercise: Flattening with guard clauses

Rewrite `can_borrow` with guard clauses so that no `if` is inside another `if`, keeping exactly the same results for every input.

In [ ]:
def can_borrow(member, book):
    if member is not None:
        if not member["banned"]:
            if book["available"]:
                if member["loans"] < 5:
                    return True
                else:
                    return False
            else:
                return False
        else:
            return False
    else:
        return False

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_flat():\n    \"flat and the same\"\n    import ast, itertools\n    for banned, avail, loans in itertools.product([True, False], [True, False], [0, 4, 5, 9]):\n        m = {\"banned\": banned, \"loans\": loans}\n        want = (not banned) and avail and loans < 5\n        assert can_borrow(m, {\"available\": avail}) == want, f\"wrong for {m}, available={avail}\"\n    assert can_borrow(None, {\"available\": True}) is False\n    fn = next(n for n in ast.walk(ast.parse(source)) if isinstance(n, ast.FunctionDef) and n.name == \"can_borrow\")\n    for outer in ast.walk(fn):\n        if isinstance(outer, ast.If):\n            for inner in ast.walk(outer):\n                assert inner is outer or not isinstance(inner, ast.If), \"no if inside another if\"")

### Exercise: Extracting functions

`invoice` does three jobs in one long body. Extract them into `line_total(qty, pence, discount)` (the line's price after a percentage discount, rounded down), `vat_on(net)` (20 per cent, rounded down) and `format_invoice(lines, net, vat)` (the text), so that `invoice` becomes a few lines calling them. The output must not change.

In [ ]:
def invoice(items):
    lines = []
    net = 0
    for name, qty, pence, discount in items:
        total = qty * pence
        total = total - total * discount // 100
        net += total
        lines.append((name, total))
    vat = net * 20 // 100
    out = []
    for name, total in lines:
        out.append(f"{name:<10}{total:>8}")
    out.append(f"{'Net':<10}{net:>8}")
    out.append(f"{'VAT':<10}{vat:>8}")
    out.append(f"{'Total':<10}{net + vat:>8}")
    return "\n".join(out)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_extract():\n    \"the same invoice, from parts\"\n    import ast\n    items = [(\"Rice\", 2, 450, 0), (\"Oil\", 1, 999, 10), (\"Salt\", 3, 120, 50)]\n    want = \"Rice           900\\nOil            900\\nSalt           180\\nNet           1980\\nVAT            396\\nTotal         2376\"\n    assert invoice(items) == want, \"got:\\n\" + invoice(items)\n    assert line_total(1, 999, 10) == 900 and line_total(3, 120, 50) == 180 and vat_on(1980) == 396\n    assert format_invoice([(\"A\", 5)], 5, 1) == \"A                5\\nNet              5\\nVAT              1\\nTotal            6\"\n    fn = next(n for n in ast.walk(ast.parse(source)) if isinstance(n, ast.FunctionDef) and n.name == \"invoice\")\n    assert fn.end_lineno - fn.lineno <= 5, \"invoice should be a few lines that call the others\"")

### Exercise: Names and numbers

Refactor `calc` without changing its results: rename it `parking_charge`, with parameters `minutes` and `is_resident`; replace its magic numbers with constants `FREE_MINUTES = 30`, `HOURLY_PENCE = 150`, `DAILY_CAP = 1200` and `RESIDENT_DISCOUNT = 50` (per cent); and give the hours a name. Delete `calc`.

In [ ]:
import math

def calc(m, r):
    if m <= 30:
        return 0
    c = min(math.ceil((m - 30) / 60) * 150, 1200)
    if r:
        c = c - c * 50 // 100
    return c

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_names():\n    \"clearer, same results\"\n    import inspect, math\n    def old(m, r):\n        if m <= 30:\n            return 0\n        c = min(math.ceil((m - 30) / 60) * 150, 1200)\n        if r:\n            c = c - c * 50 // 100\n        return c\n    for m in [0, 30, 31, 90, 91, 600, 2000]:\n        for r in [False, True]:\n            assert parking_charge(m, r) == old(m, r), f\"different for {m} minutes, resident={r}\"\n    assert list(inspect.signature(parking_charge).parameters) == [\"minutes\", \"is_resident\"] and \"calc\" not in ns\n    assert (FREE_MINUTES, HOURLY_PENCE, DAILY_CAP, RESIDENT_DISCOUNT) == (30, 150, 1200, 50)\n    body = source.split(\"def parking_charge\", 1)[1]\n    for n in [\"30\", \"150\", \"1200\", \"50\"]:\n        assert n not in body, f\"use the constant instead of {n}\"")

### Bug hunt: A boundary moved

Someone extracted the age check into `is_child`, and child tickets now stop a year early: a 15-year-old pays the adult price. The original rule was "under 16 is a child". Fix `is_child` so the program prints `[600, 600, 1000]`.

In [ ]:
def is_child(age):
    return age < 15

def ticket(age):
    return 600 if is_child(age) else 1000

print([ticket(a) for a in [4, 15, 16]])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the original rule\"\n    expect_output(\"[600, 600, 1000]\")")

### Bug hunt: Guards in the wrong order

While turning nested `if`s into guard clauses, the order of the checks changed, and now staff over the spending limit get refused instead of being allowed (staff have no limit). The original rule: staff always may; others may if the amount is at most 5000. Fix the order so it prints `[True, True, False, True]`.

In [ ]:
def may_spend(person, amount):
    if amount > 5000:
        return False
    if person["staff"]:
        return True
    return True

people = [({"staff": True}, 100), ({"staff": True}, 9000), ({"staff": False}, 9000), ({"staff": False}, 5000)]
print([may_spend(p, a) for p, a in people])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"staff first\"\n    expect_output(\"[True, True, False, True]\")")

### Bug hunt: A variable extracted too early

"Extract variable" moved `remaining` out of the loop, so it's worked out once, before any seats are booked, and the program overbooks the coach: it prints `[True, True, True]` for a coach with 2 free seats. Fix it so each booking sees the current number of free seats, printing `[True, True, False]`.

In [ ]:
def book_all(capacity, requests):
    booked = 0
    results = []
    remaining = capacity - booked
    for seats in requests:
        if seats <= remaining:
            booked += seats
            results.append(True)
        else:
            results.append(False)
    return results

print(book_all(2, [1, 1, 1]))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"no overbooking\"\n    expect_output(\"[True, True, False]\")\n    assert book_all(5, [3, 3, 2]) == [True, False, True]")

### Puzzle: Guard clauses

Arrange the lines into a function that uses guard clauses to give a parcel's status message. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Refactoring or not?

Is each change a refactoring (structure only, same behaviour)? Answer `True` or `False`.

In [ ]:
# p: Renaming a function's local variable total to net_total everywhere in the function.
# q: Changing a function so that it raises ValueError for negative input instead of returning 0.
# r: Moving three lines into a new function and calling it from the same place.
# s: Making a slow function faster while it returns the same results.
# t: Replacing a list of if/elif cases with a dictionary lookup that gives the same results, including the error for unknown keys.

p = ___
q = ___
r = ___
s = ___
t = ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert (p, q, r) == (True, False, True), \"p to r\"\n    assert (s, t) == (True, True)")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: refactoring a ticket price

`price` works out a cinema ticket's price, and it's hard to read. Refactor it so that its behaviour is **exactly** the same for every combination of inputs the checks try, while meeting these structural goals:

- named constants instead of every magic number (there are eight distinct numbers);
- no `if` nested inside another `if` in any function;
- at least two extracted helper functions, named `base_price(age, is_3d)` and `discount(price, day, is_member)`, used by `price`;
- `price` itself no longer than 8 lines.

Don't change the behaviour, even where it looks odd (members get the Tuesday discount too, but not both).

In [ ]:
def price(age, day, is_3d, is_member):
    if age < 0:
        raise ValueError("bad age")
    if age < 3:
        return 0
    else:
        if age < 16:
            p = 650
        else:
            if age >= 65:
                p = 700
            else:
                p = 1100
        if is_3d:
            p = p + 250
        if day == "Tue":
            p = p - p * 30 // 100
        else:
            if is_member:
                p = p - p * 10 // 100
        return p

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def old(age, day, is_3d, is_member):\n    if age < 0:\n        raise ValueError(\"bad age\")\n    if age < 3:\n        return 0\n    else:\n        if age < 16:\n            p = 650\n        else:\n            if age >= 65:\n                p = 700\n            else:\n                p = 1100\n        if is_3d:\n            p = p + 250\n        if day == \"Tue\":\n            p = p - p * 30 // 100\n        else:\n            if is_member:\n                p = p - p * 10 // 100\n        return p\ndef test_same():\n    \"exactly the same behaviour\"\n    import itertools\n    for age, day, d3, mem in itertools.product([0, 2, 3, 15, 16, 64, 65, 90], [\"Mon\", \"Tue\", \"Sat\"], [False, True], [False, True]):\n        assert price(age, day, d3, mem) == old(age, day, d3, mem), f\"different for age={age}, day={day}, 3D={d3}, member={mem}\"\n    try:\n        price(-1, \"Mon\", False, False)\n        assert False, \"a negative age should still raise ValueError\"\n    except ValueError:\n        pass\ndef test_structure():\n    \"the refactoring goals\"\n    import ast\n    tree = ast.parse(source)\n    funcs = {n.name: n for n in ast.walk(tree) if isinstance(n, ast.FunctionDef)}\n    assert \"base_price\" in funcs and \"discount\" in funcs, \"extract base_price and discount\"\n    assert base_price(70, True) == 950 and discount(1000, \"Tue\", True) == 700 and discount(1000, \"Mon\", True) == 900\n    for f in funcs.values():\n        for outer in ast.walk(f):\n            if isinstance(outer, ast.If):\n                for inner in ast.walk(outer):\n                    assert inner is outer or not isinstance(inner, ast.If), f\"{f.name} has an if inside an if\"\n    p = funcs[\"price\"]\n    assert p.end_lineno - p.lineno + 1 <= 8, \"price should be 8 lines or fewer\"\n    numbers = [n.value for n in ast.walk(p) if isinstance(n, ast.Constant) and isinstance(n.value, int) and not isinstance(n.value, bool)]\n    assert set(numbers) <= {0}, f\"name these numbers as constants: {numbers}\"\n    for name in [\"base_price\", \"discount\"]:\n        nums = [n.value for n in ast.walk(funcs[name]) if isinstance(n, ast.Constant) and isinstance(n.value, int) and not isinstance(n.value, bool)]\n        assert set(nums) <= {0, 100}, f\"{name} uses magic numbers: {nums}\"")

## Warm-up

Two very different-looking functions, and the same answers for every case. The second is the first after **refactoring**: renamed, with its magic numbers named and its nested `if`s flattened, and with exactly the same behaviour. Today is about getting from the first to the second safely.

## What refactoring is

Martin Fowler's book *Refactoring* (1999, second edition 2018) defines it as "a change made to the internal structure of software to make it easier to understand and cheaper to modify without changing its observable behaviour." Two parts of that definition matter most:

- **Without changing observable behaviour.** A refactoring changes how the code is arranged, never what it does. Same inputs, same outputs, same errors.
- **Easier to understand and cheaper to modify.** It has a purpose: to make the next change, or the next reader, easier.

Kent Beck describes programming as wearing **two hats**: while adding a feature, you add behaviour and don't restructure; while refactoring, you restructure and don't add behaviour. Swapping hats often is fine, but wearing both at once is how bugs hide: if behaviour changes during a "refactoring", you can't tell whether it was the new feature or a mistake.

## Tests first

You can only be sure behaviour hasn't changed if something checks it. Before refactoring code without tests, write tests that record what it **does now**: Michael Feathers calls these **characterisation tests** (in *Working Effectively with Legacy Code*). They're different from Week 24's tests in one important way: they record actual behaviour, even if it looks wrong.

Some of these answers look like bugs: lower-case `"uk"` is charged as abroad, and very heavy parcels give `None` rather than an error. A characterisation test records them anyway. The aim is "the refactored code does exactly what the old code did", not "the code is right". If an oddity is a real bug, fix it **separately**, wearing the other hat, with its own test that's changed on purpose.

Turning the exploration into tests is straightforward:

```python
from legacy import shipping_cost

def test_boundaries():
    assert shipping_cost(1000, "UK") == 300
    assert shipping_cost(1001, "UK") == 400

def test_lower_case_country_is_abroad():      # odd, but it's what happens
    assert shipping_cost(500, "uk") == 1500

def test_too_heavy_gives_none():
    assert shipping_cost(20001, "UK") is None
    assert shipping_cost(25000, "FR", express=True) is None
```

## A catalogue of small steps

Refactorings have names, which makes them easier to talk about and to do safely. Here are the most useful, each with a before and after:

| Refactoring | Before | After |
|---|---|---|
| Rename | `def f(d, b):` | `def late_fee(days_late, books):` |
| Replace magic number with constant | `x * 20` | `x * DAILY_FEE` |
| Extract variable | `if t.hour >= 9 and t.hour < 17 and t.weekday() < 5:` | `in_office_hours = ...` then `if in_office_hours:` |
| Extract function | ten lines inside a loop | a named function called from the loop |
| Inline | a variable or function that adds nothing: `result = x; return result` | `return x` |
| Guard clauses | nested `if`s around the main path | handle special cases first and `return` |
| Introduce parameter object | `book(name, email, phone, ...)` | `book(contact, ...)` (Day 1) |
| Replace conditional with polymorphism or a table | `if kind == ...` chains | classes or a dictionary (Day 2, Day 4) |
| Split loop | one loop doing two jobs | two loops, each with one job |

**Guard clauses** deserve a closer look, because deep nesting is one of the commonest causes of hard-to-read code:

Each guard deals with one special case and gets it out of the way, so the main rule sits at the bottom with no nesting. **Extract function** is the other workhorse: when a block of code needs a comment to say what it does, it usually wants to be a function whose name says it.

## One step at a time

The safe way to refactor is a loop:

1. Make sure the tests pass.
2. Make **one** small change from the catalogue.
3. Run the tests.
4. If they pass, commit (Week 21: `git commit` gives you a point to come back to). If they fail, **undo** (`git restore`) rather than debugging: the step was too big, so try a smaller one.

Here's a refactoring of the warm-up's `f`, one step at a time, with the tests run after each:

| Step | Change |
|---|---|
| 1 | Rename `f`, `d`, `b` to `late_fee`, `days_late`, `books` |
| 2 | Replace `20` with `DAILY_FEE` and `500` with `FEE_CAP` |
| 3 | Replace the inner `if x > FEE_CAP: x = FEE_CAP` with `x = min(x, FEE_CAP)` |
| 4 | Turn the nested `if`s into a guard clause: `if days_late <= 0 or books <= 0: return 0` |
| 5 | Inline `x`: `return min(days_late * DAILY_FEE * books, FEE_CAP)` |

Every step is small enough to get right, and boring enough to do quickly. The result is the second function in the warm-up. Editors can do some steps (rename, extract function, extract variable) automatically and safely: in VS Code, select the code and look for "Refactor" in the right-click menu.

## When to refactor

- **Before a change** ("preparatory refactoring"): when a new feature is awkward to add, first refactor so that it's easy, then add it. Kent Beck: "make the change easy (warning: this may be hard), then make the easy change."
- **When reading code**: if you had to work hard to understand something, rename or extract to record what you learned, so the next reader doesn't have to.
- **The rule of three** (Day 4): on the third repetition, extract.
- **Not** when there are no tests and no time to write any, when the code is about to be deleted, or in the middle of fixing an urgent bug (fix it, commit, then refactor).

Refactoring isn't a separate project scheduled for "later": it's small and continuous, done as part of everyday work, which is why it rarely needs permission or a plan.

## Practice

## Quiz

## Daily challenge

## Before you go

Choose a function of your own that you find hard to read. Write a few characterisation tests for it, then refactor it in at least three small named steps, running the tests after each. Tomorrow is the Forge: a whole tangled program, redesigned with this week's principles.